Loads the preprocessed AnnDatas + the three clustered AnnDatas written by
tutorial 3, and runs the full STORM benchmark via
:func:`storm.evaluation.benchmark`. The same metric functions are also
exposed individually under :mod:`storm.evaluation` if you want to call
them à la carte.

Four families of metrics (see :mod:`storm.evaluation` for definitions):

* **Multi-omics integration** — FOSCTTM, MLISI, Consistency ARI.
* **Batch / timepoint integration** — PCR (needs ``scib``), BLISI.
* **Cell-type recovery** — NMI vs. each ground-truth ``*_clusters``
  column (needs ``scib``).
* **Niche coherence + spatial conservation** — LISI, ASW, MAP, Global
  Moran's I (needs ``squidpy``), CLISIS.

**Conda environment note.** PCR/NMI use ``scib`` and Moran's I uses
``squidpy``; the GNN training env does not ship those by default. Run
this notebook from an env that has them (e.g. ``SpatialGlue`` on this
cluster) — :func:`storm.evaluation.benchmark` raises a clear
``ImportError`` naming the missing dep if you forget.

# STORM tutorial 4: evaluation metrics

Four families of metrics, all derived from the shared latent space
(``X_storm``) and the cluster labels written by tutorial 3. Everything
runs through :func:`storm.evaluation.benchmark` so the harness is
reusable on any analysis script — not just this tutorial.

In [1]:
import os
import sys

_REPO_ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), os.pardir)) \
    if "__file__" in globals() else os.path.abspath(os.path.join(os.getcwd(), os.pardir))
for _candidate in (_REPO_ROOT, os.getcwd()):
    if os.path.isdir(os.path.join(_candidate, "storm")) and _candidate not in sys.path:
        sys.path.insert(0, _candidate)
        break

import anndata as ad
import pandas as pd

import storm
from storm.clustering import concat_clustering
from storm.evaluation import benchmark
from storm.models import load_model

/gpfs/gibbs/project/zhao/xc384/conda_envs/storm/lib/python3.10/site-packages/torch/cuda/__init__.py:56: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


## 1. Parameters and inputs

Defaults assume tutorials 1-3 have all been run. ``TRAINED_DILL`` is the
checkpoint to evaluate — point it at your full-length training run for
numbers that match the paper.

In [2]:
TUTORIAL_OUT = "../artifacts/storm_tutorial"
PREP_DIR = f"{TUTORIAL_OUT}/preprocessed"
CLUSTER_DIR = f"{TUTORIAL_OUT}/clustered"
METRICS_DIR = f"{TUTORIAL_OUT}/metrics"
os.makedirs(METRICS_DIR, exist_ok=True)

PREP_RNA = f"{PREP_DIR}/rna_preprocessed.h5ad"
PREP_ATAC = f"{PREP_DIR}/atac_preprocessed.h5ad"
TRAINED_DILL = "../dill_files/FINE_storm_P21P22_reftarg160_2tmp_RNAr_peak_source_repro.dill"

# Names line up with what tutorial_3_clustering.py writes.
N_CLUSTERS_CONCAT = 16
CONCAT_RNA = f"{CLUSTER_DIR}/rna_concat{N_CLUSTERS_CONCAT}.h5ad"
CONCAT_ATAC = f"{CLUSTER_DIR}/atac_concat{N_CLUSTERS_CONCAT}.h5ad"

METRICS_CSV = f"{METRICS_DIR}/storm_metrics.csv"
SAMPLES = ["P21", "P22"]

## 2. Load preprocessed + clustered AnnDatas and the trained model

Encoding through the trained ``DataEncoder`` rewrites ``X_storm`` (so the
evaluation reflects the *current* checkpoint, not whatever was cached
in the clustered h5ads).

In [3]:
rna = ad.read_h5ad(PREP_RNA)
atac = ad.read_h5ad(PREP_ATAC)

storm_model = load_model(TRAINED_DILL)
rna.obsm["X_storm"] = storm_model.encode_data("rna", rna)
atac.obsm["X_storm"] = storm_model.encode_data("atac", atac)

# Modality-specific (CONCAT) cluster labels — required by Consistency ARI.
if os.path.exists(CONCAT_RNA) and os.path.exists(CONCAT_ATAC):
    rna_concat = ad.read_h5ad(CONCAT_RNA)
    atac_concat = ad.read_h5ad(CONCAT_ATAC)
    rna.obs["domain"] = rna_concat.obs["domain"].values
    atac.obs["domain"] = atac_concat.obs["domain"].values
    print(f"Loaded CONCAT cluster labels from {CONCAT_RNA}")
else:
    print("Concat-clustered artifacts missing — recomputing on the fly.")
    concat_clustering(rna, atac, n_clusters=N_CLUSTERS_CONCAT,
                      key="X_storm", method="louvain_merge")

# Joint AnnData — needed for PCR / NMI / LISI / ASW / MAP / Moran's I / CLISIS.
joint = None
for fname in sorted(os.listdir(CLUSTER_DIR) if os.path.exists(CLUSTER_DIR) else []):
    if fname.startswith("joint_clustering") and fname.endswith(".h5ad"):
        joint = ad.read_h5ad(f"{CLUSTER_DIR}/{fname}")
        print(f"Loaded JOINT clustering: {fname} ({joint.n_obs} cells)")
        # Re-derive X_storm from the current checkpoint so the metrics
        # reflect the loaded model rather than a stale embedding.
        joint.obsm["X_storm"] = rna.obsm["X_storm"] + atac.obsm["X_storm"]
        break
if joint is None:
    print("No JOINT clustering h5ad found — joint metrics will be skipped.")

[INFO] autodevice: Using GPU 0 as computation device.


INFO:autodevice:Using GPU 0 as computation device.
/gpfs/gibbs/project/zhao/xc384/conda_envs/storm/lib/python3.10/abc.py:119: FutureWarning: SparseDataset is deprecated and will be removed in late 2024. It has been replaced by the public classes CSRDataset and CSCDataset.

For instance checks, use `isinstance(X, (anndata.experimental.CSRDataset, anndata.experimental.CSCDataset))` instead.

For creation, use `anndata.experimental.sparse_dataset(X)` instead.

  return _abc_instancecheck(cls, instance)


Loaded CONCAT cluster labels from ../artifacts/storm_tutorial/clustered/rna_concat16.h5ad
Loaded JOINT clustering: joint_clustering16.h5ad (11324 cells)


## 3. Run the benchmark

:func:`storm.evaluation.benchmark` runs every metric family on the
inputs you supply and returns one tidy DataFrame. ``include`` /
``exclude`` filter by metric name (substring, case-insensitive) so
you can opt out of the metrics whose deps you don't have.

Try ``exclude=["Joint PCR", "Joint NMI", "Joint Global Morans I"]``
to run from the lightweight GNN environment without ``scib`` or
``squidpy``.

In [4]:
metrics_df = benchmark(
    rna, atac, joint=joint,
    embed_key="X_storm",
    domain_key="domain",
    sample_key="timepoint",
    samples=SAMPLES,
    ref_keys=("RNA_clusters", "ATAC_clusters"),
)

pd.set_option("display.max_rows", None)
print(metrics_df.to_string(index=False))
metrics_df.to_csv(METRICS_CSV, index=False)
print(f"\nWrote {METRICS_CSV}")

/gpfs/gibbs/project/zhao/xc384/conda_envs/storm/lib/python3.10/site-packages/scanpy/preprocessing/_pca.py:385: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can be called through mask_var=None
  warn(msg, FutureWarning)
/gpfs/gibbs/project/zhao/xc384/conda_envs/storm/lib/python3.10/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/gpfs/gibbs/project/zhao/xc384/conda_envs/storm/lib/python3.10/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package i

  0%|          | 0/100 [00:00<?, ?/s]

  0%|          | 0/100 [00:00<?, ?/s]

  0%|          | 0/100 [00:00<?, ?/s]

                                Metric  n_clusters  Sample  Value
                               FOSCTTM          16 Overall 0.9840
                               FOSCTTM          16     P21 0.9832
                               FOSCTTM          16     P22 0.9836
                                 MLISI          16 Overall 0.0511
                                 MLISI          16     P21 0.0484
                                 MLISI          16     P22 0.0509
                           Consistency          16     P21 0.6778
                           Consistency          16     P22 0.6573
                           Consistency          16 Overall 0.6675
                             Joint PCR          16 Overall 0.9899
                           Joint BLISI          16 Overall 0.0258
           Joint NMI with RNA_clusters          16     P21 0.4730
           Joint NMI with RNA_clusters          16     P22 0.3847
           Joint NMI with RNA_clusters          16 Overall 0.2787
          

### A wider, easier-to-read view

Pivot so each row is one metric and each column is one sample.

In [5]:
pivot = metrics_df.pivot_table(
    index="Metric", columns="Sample", values="Value", aggfunc="first",
)
# Put the per-sample columns first, "Overall" last.
sample_cols = [c for c in pivot.columns if c != "Overall"] + (
    ["Overall"] if "Overall" in pivot.columns else []
)
print(pivot[sample_cols].to_string())

Sample                                     P21     P22  Overall
Metric                                                         
Consistency                             0.6778  0.6573   0.6675
FOSCTTM                                 0.9832  0.9836   0.9840
Joint BLISI                                NaN     NaN   0.0258
Joint CLISIS with ATAC_clusters         0.7480  0.7661      NaN
Joint CLISIS with RNA_clusters          0.7531  0.8197      NaN
Joint Cell Type ASW with ATAC_clusters  0.5792  0.5337   0.4973
Joint Cell Type ASW with RNA_clusters   0.5848  0.5499   0.4981
Joint Global Moran's I                  0.8331  0.8964   0.8608
Joint LISI with ATAC_clusters           0.9218  0.9059   0.8896
Joint LISI with RNA_clusters            0.9381  0.9286   0.8995
Joint MAP with ATAC_clusters            0.6625  0.5430   0.4899
Joint MAP with RNA_clusters             0.6746  0.5690   0.4552
Joint NMI with ATAC_clusters            0.4208  0.3896   0.3288
Joint NMI with RNA_clusters             

## 4. Calling individual metric functions

Every entry in :mod:`storm.evaluation` is independently usable —
:func:`benchmark` is just a thin wrapper that orchestrates them. For
example, to grab just the multi-omics integration scores:

```python
from storm.evaluation import foscttm_paired, mlisi, consistency_ari
foscttm_paired(rna, atac, samples=["P21", "P22"])
mlisi(rna, atac, samples=["P21", "P22"])
consistency_ari(rna, atac, samples=["P21", "P22"])
```

Or call a single niche-coherence metric on the joint AnnData:

```python
from storm.evaluation import lisi, clisis
lisi(joint, annotation_key="RNA_clusters", samples=["P21", "P22"])
clisis(joint, cell_type_key="RNA_clusters", samples=["P21", "P22"])
```

## 5. Interpreting the numbers

* **FOSCTTM ≈ 1.0** and **MLISI ≈ 1.0** mean STORM has fully bridged
  the RNA / ATAC distributions in the latent space — paired RNA / ATAC
  cells co-localise and the latent space is well-mixed across
  modalities.
* **Consistency ARI** in the 0.5+ range means RNA-derived and
  ATAC-derived CONCAT clusters agree on most cells.
* **NMI / ARI vs ground truth** below 0.3 with a 5-epoch demo model is
  expected. After a full-length training run (~190 fine-tune epochs),
  these should approach the values reported in the paper.

Wherever the tutorial-short numbers are weak, retrain at the full
epoch budget and point ``TRAINED_DILL`` here at the resulting
``.dill`` — every metric above re-runs against whichever checkpoint
is loaded.